# 11. MCP (Model Context Protocol)

> **MCP는 "도구들을 LLM에 표준 방식으로 연결해주는 프로토콜"입니다.**

## 학습 목표
- MCP가 왜 필요한지 안다
- MCP 클라이언트로 외부 도구 서버에 연결한다
- 가져온 도구를 LangChain 에이전트에 바로 꽂아 쓴다
- 여러 MCP 서버를 동시에 연결할 수 있다


## 사전 준비 

이 노트북은 두 개의 MCP 서버에 연결합니다. **노트북을 실행하기 전에**
별도 터미널 두 개에서 각각 서버를 띄워두세요.

### 🖥️ 터미널 1 — Math 서버

cd 05-mcp   
uv run python math_server.py

→ `http://localhost:8001/mcp` 에서  시작

### 🖥️ 터미널 2 — Weather 서버

cd 05-mcp  
uv run python get_weather.py  

→ `http://localhost:8000/mcp` 에서  시작

### ✅ 정상 신호
두 터미널에 각각 이런 메시지가 떠야 합니다:
\```
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
\```

 WinError 10054 = "원격 호스트가 연결을 강제 종료함" - 무시

MCP 클라이언트가 도구 호출을 다 끝내고 정상적으로 SSE 스트림을 닫는 과정에서 Windows의 asyncio Proactor 이벤트 루프가 소켓 정리를 시도하는데, 이미 닫힌 소켓에 shutdown()을 한 번 더 부르면서 뱉는 경고.



In [1]:
# .env 파일에서 API 키(OPENAI_API_KEY 등)를 로드합니다.
from dotenv import load_dotenv
import os
load_dotenv(override=True)

print("환경 준비 완료.")

환경 준비 완료.


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

LangSmith tracing ON — project: day1-labs-test


In [3]:
# OpenTelemetry 컨텍스트 관련 경고 로그를 억제합니다.
# MCP 클라이언트가 내부적으로 OpenTelemetry를 사용하면서 불필요한 경고가 출력될 수 있습니다.
import logging
logging.getLogger("opentelemetry.context").setLevel(logging.CRITICAL)

## 11.2 MCP 개념

**MCP(Model Context Protocol)**는 외부 도구와 컨텍스트를 **표준화된 방식**으로 LLM에 제공하기 위한 오픈 프로토콜입니다.

### 아키텍처 구성 요소

| 구성 요소 | 역할 | 예시 |
|----------|------|------|
| **MCP 서버** | 도구, 리소스, 프롬프트를 노출 | 파일 시스템 서버, DB 서버, API 래퍼 |
| **MCP 클라이언트** | 서버에 연결하여 도구를 가져옴 | `MultiServerMCPClient` |
| **호스트** | 클라이언트를 관리하고 LLM과 연결 | LangChain 에이전트, IDE |

### 핵심 리소스 타입

- **Tools**: 에이전트가 호출할 수 있는 실행 가능한 함수
- **Resources**: 파일, DB 레코드 등의 데이터 (LangChain Blob 객체로 변환)
- **Prompts**: 재사용 가능한 프롬프트 템플릿

### 왜 MCP인가?

MCP 이전에는 각 도구마다 개별적으로 연결 코드를 작성해야 했습니다. MCP는 이를 **하나의 표준 프로토콜**로 통합하여:
- 도구 제공자는 한 번만 MCP 서버를 구현하면 됩니다
- LLM 호스트는 MCP 클라이언트 하나로 모든 도구에 접근할 수 있습니다
- 생태계 전체에서 도구를 재사용할 수 있습니다


In [4]:
# 서버가 진짜 떠 있는지 먼저 확인 — 안 떠 있으면 여기서 끝내고 위로!
import httpx

for name, url in [("math   ", "http://localhost:8001/mcp"),
                  ("weather", "http://localhost:8000/mcp")]:
    try:
        # MCP 서버는 GET을 직접 안 받지만 연결만 되면 OK
        httpx.get(url, timeout=3)
        print(f" {name} 서버: 살아있음")
    except Exception as e:
        print(f" {name} 서버: 안 떠있음! 터미널에서 실행 확인 → {type(e).__name__}")


 math    서버: 살아있음
 weather 서버: 살아있음


## 다중 MCP 서버 연결

`MultiServerMCPClient`로 **한 곳에 연결**합니다.

### 흐름 그림
\```
[Jupyter 노트북]  
      ↓ MultiServerMCPClient  
      ├──→ math 서버    (8001) → add, multiply  
      └──→ weather 서버 (8000) → get_weather  
\```

### 핵심 포인트 3가지
1. **dict로 등록** — 키 이름(`"math"`, `"weather"`)이 서버 식별용
2. **`get_tools()`가 모든 서버의 도구를 한 번에 모아옴** — 에이전트는 어디서 왔는지 신경 안 씀
3. **`await` 필수** — MCP는 비동기 기반


In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# ──────────────────────────────────────────────────────────────────
# 배달앱(client)에 두 음식점을 등록
# ──────────────────────────────────────────────────────────────────
client = MultiServerMCPClient(
    {
        "math": {                                        
            "transport": "streamable_http",
            "url": "http://localhost:8001/mcp",
        },
        "weather": {                                     
            "transport": "streamable_http",
            "url": "http://localhost:8000/mcp",
            # 상용 MCP 서버 인증이 필요하면:
            # "headers": {"Authorization": "Bearer YOUR_TOKEN"},
        }
    }
)

# 두 결과 한 번에 가져오기 (await 필수)
tools = await client.get_tools()

print(f"✓ 가져온 도구 {len(tools)}개: {[t.name for t in tools]}")


✓ 가져온 도구 3개: ['add', 'multiply', 'get_weather']


## 잠깐 멈춰서 보기


- `add`, `multiply`는 math 서버에서 옴
- `get_weather`는 weather 서버에서 옴
- **에이전트 입장에선 그냥 평범한 도구 3개**




In [6]:
from langchain.agents import create_agent

# ──────────────────────────────────────────────────────────────────
# 에이전트에 도구 꽂기 — MCP 도구든 직접 만든 도구든 똑같이 취급
# ──────────────────────────────────────────────────────────────────
agent = create_agent("openai:gpt-5.4-mini", tools)

# ── 테스트 1: 수학 (add → multiply 자동 호출) ──
math_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "3에 5더하고 12곱하면?"}]},
    config=lf_config
)

# ── 테스트 2: 날씨 (get_weather 자동 호출) ──
weather_response = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "뉴욕 날씨는?"}]},
    config=lf_config
)

print("[Math 결과]")
print(math_response['messages'][-1].content)
print("\n[Weather 결과]")
print(weather_response['messages'][-1].content)


[Math 결과]
3에 5를 더하면 8이고, 그걸 12와 곱하면 **96**입니다.

[Weather 결과]
뉴욕 날씨는 **맑음**입니다.


## 결과 해석

### 정상 출력 예시
\```
[Math 결과]
(3 + 5) × 12 = 96

[Weather 결과]
It's always sunny in New York
\```

에이전트가 자동으로:
1. `add(3, 5)` 호출 → 8
2. `multiply(8, 12)` 호출 → 96
3. 자연어로 정리해서 답변

→ **우리가 도구 호출 코드를 한 줄도 안 짰다는 게 핵심!**   
도구 목록만 넘기면 에이전트가 알아서 합니다.

## 자주 만나는 에러

| 에러 | 원인 | 해결 |
|---|---|---|
| `Connection refused` | 서버 안 떠 있음 | 터미널에서 서버 실행 확인 |
| `WinError 10048` | 포트 충돌 | `taskkill /PID <num> /F` |
| `fileno` 에러 | stdio 잔재 / 커널 캐시 | 커널 재시작 → Run All |
| 코드 바꿨는데 에러 동일 | 셀 저장 안 됨 | `Ctrl+S` → 커널 재시작 |
